In [1]:
!pip install -q sentence-transformers scikit-learn numpy

In [2]:
import re
samples = ["AI, search!", "I can't log in.", "help@example.com", "CS4944 version 2.1", "", "میں یونیورسٹی جا رہا ہوں"]
for text in samples:
    ws = text.split()
    words = re.findall(r"\b\w+\b", text, flags=re.UNICODE)
    print(repr(text))
    print("Whitespace:", ws, len(ws))
    print("Word-like:", words, len(words))
from sentence_transformers import SentenceTransformer
import time
start = time.perf_counter()
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
print("Model load seconds:", round(time.perf_counter() - start, 2))
print("Model token limit:", model.max_seq_length)
for text in samples[:2]:
    tokens = model.tokenizer.tokenize(text)
    print(repr(text), "Model tokens:", tokens, len(tokens))

'AI, search!'
Whitespace: ['AI,', 'search!'] 2
Word-like: ['AI', 'search'] 2
"I can't log in."
Whitespace: ['I', "can't", 'log', 'in.'] 4
Word-like: ['I', 'can', 't', 'log', 'in'] 5
'help@example.com'
Whitespace: ['help@example.com'] 1
Word-like: ['help', 'example', 'com'] 3
'CS4944 version 2.1'
Whitespace: ['CS4944', 'version', '2.1'] 3
Word-like: ['CS4944', 'version', '2', '1'] 4
''
Whitespace: [] 0
Word-like: [] 0
'میں یونیورسٹی جا رہا ہوں'
Whitespace: ['میں', 'یونیورسٹی', 'جا', 'رہا', 'ہوں'] 5
Word-like: ['میں', 'یونیورسٹی', 'جا', 'رہا', 'ہوں'] 5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model load seconds: 7.7
Model token limit: 256
'AI, search!' Model tokens: ['ai', ',', 'search', '!'] 4
"I can't log in." Model tokens: ['i', 'can', "'", 't', 'log', 'in', '.'] 7


In [3]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer


class Retriever:
    def __init__(self, records, backend="tfidf"):
        if not records or any(not r["text"].strip() for r in records):
            raise ValueError("Provide nonempty records")
        self.records, self.backend = records, backend
        texts = [r["text"] for r in records]
        if backend == "tfidf":
            self.vectorizer = TfidfVectorizer()
            self.matrix = self.vectorizer.fit_transform(texts)
        elif backend == "embedding":
            self.model = model
            self.check_length(texts)
            self.matrix = self.model.encode(texts, normalize_embeddings=True)
        else:
            raise ValueError("Use tfidf or embedding")

    def check_length(self, texts):
        for text in texts:
            n = len(self.model.tokenizer(text, truncation=False)["input_ids"])
            if n > self.model.max_seq_length:
                raise ValueError("Input exceeds encoder token limit")

    def search(self, query, k=3, filters=None):
        if not query.strip() or k <= 0:
            return []
        if self.backend == "tfidf":
            q = self.vectorizer.transform([query])
            if q.nnz == 0:
                return []
            scores = (q @ self.matrix.T).toarray()[0]
        else:
            self.check_length([query])
            q = self.model.encode([query], normalize_embeddings=True)[0]
            scores = self.matrix @ q
        allowed = [i for i, r in enumerate(self.records) if all(r.get(key) == value for key, value in (filters or {}).items())]
        ranked = sorted(allowed, key=lambda i: (-float(scores[i]), i))
        return [(self.records[i], float(scores[i])) for i in ranked[:k]]


def show_hits(hits):
    if not hits:
        print("No matching records.")
    for record, score in hits:
        print(record["id"], f"score={score:.3f}")
        print(record["text"])


FAQ = [
    {"id": "faq:password", "question": "How do I reset my password?", "answer": "Use the recovery link on the login page.", "source": "it_help.txt"},
    {"id": "faq:submit", "question": "Where do I submit my assignment?", "answer": "Use the assignment upload page in the learning portal.", "source": "assessment.txt"},
    {"id": "faq:late", "question": "Can illness justify a late submission?", "answer": "Request an extension with medical evidence before the deadline.", "source": "handbook.txt"},
    {"id": "faq:receipt", "question": "How do I check my assignment receipt?", "answer": "Open the learning portal and check your submission history.", "source": "portal_guide.txt"},
    {"id": "faq:blank", "question": "Why does my printer print blank pages?", "answer": "Check ink levels and run the cleaning utility.", "source": "it_help.txt"},
]


def build_bot(backend="tfidf"):
    records = [dict(r, text=r["question"] + " " + r["answer"]) for r in FAQ]
    return Retriever(records, backend)


def reply(engine, query):
    if not query.strip():
        return "Please enter a question."
    for r in FAQ:
        if query.strip().casefold() == r["question"].casefold():
            return r["answer"] + " [source: " + r["source"] + "]"
    hits = engine.search(query)
    if not hits:
        return "No matching FAQ. Please contact the help desk."
    lines = ["Possible matches; choose or rephrase your question:"]
    for r, score in hits:
        lines.append(r["question"] + f" (similarity {score:.3f})")
    return "\n".join(lines)


engine = build_bot()
print(reply(engine, "How do I reset my password?"))
print(reply(engine, "password recovery"))
print(repr(reply(engine, "")))
print(reply(engine, "volcano geology"))
assert "it_help.txt" in reply(engine, "How do I reset my password?")
assert reply(engine, "") == "Please enter a question."
assert "No matching FAQ" in reply(engine, "volcano geology")
assert "Possible matches" in reply(engine, "password recovery")
print("FAQ behaviour checks passed")

Use the recovery link on the login page. [source: it_help.txt]
Possible matches; choose or rephrase your question:
How do I reset my password? (similarity 0.456)
Where do I submit my assignment? (similarity 0.000)
Can illness justify a late submission? (similarity 0.000)
'Please enter a question.'
No matching FAQ. Please contact the help desk.
FAQ behaviour checks passed


In [4]:
docs = {
  "late": "Late submission is normally penalized. Illness extensions require medical evidence before the deadline. Every assignment in this course has a fixed deadline published in the learning portal at the start of the semester. Work submitted after the deadline loses ten percent of its marks for each day it is late, and work submitted more than five days late receives no marks at all. A student who was ill during the deadline period may request an extension, but the request must reach the course coordinator before the original deadline passes. The request must include a medical certificate from a registered doctor that states the dates on which the student was unfit to study. A certificate submitted after the deadline does not remove the late penalty. Extensions are granted only for the number of days covered by the certificate, and the new deadline is set by the coordinator. Students should keep a copy of their medical evidence and the approval message. If an extension is refused, the student may appeal to the department within seven days. Group assignments follow the same rule, and one member being ill does not extend the deadline for the whole group unless the coordinator agrees. The portal records the exact time of every submission, so the timestamp decides whether work is late.",
  "password": "Reset a forgotten password using the recovery link. Never share recovery codes. Every student account uses the university email address as its username, and the password is set by the student during the first login. If you forget your password, open the login page and click the recovery link. A reset message is sent to your registered email address, and the link inside it stays valid for one hour. Choose a new password with at least eight characters, including one number and one capital letter. Do not use your name, your roll number, or your date of birth as a password. After three failed login attempts, the account locks for thirty minutes, and you must wait before trying again. If the account stays locked, contact the IT help desk with your student ID card. Recovery codes are sent only to your own email, and no staff member will ever ask you for a code or a password. Keep your recovery codes in a safe place, because anyone who has them can take over your account. Change your password at least once every semester, and sign out on shared computers.",
  "assignment": "Upload assignments through the learning portal. Submit the final file as a PDF. Each lab task in this course has its own upload link, and the link closes at the deadline. Name your file with your roll number followed by the lab number, for example 025_lab6.pdf. The portal accepts only one file per task, so check your file before you press submit. A file uploaded in the wrong format is marked as missing, and the teacher is not required to accept it by email. After uploading, open the submitted file once to confirm that the text and figures are complete. The portal shows a receipt number for every successful submission, and you should save it. If the portal is slow near the deadline, do not wait until the last hour, because late uploads are penalized under the assessment rules. Drafts saved in the portal are not submissions. Only the file you press submit on is graded. If two files are uploaded by mistake, the latest one replaces the earlier one. Keep a copy of every submitted file on your own computer until the semester ends.",
  "contractor": "Contractors can work remotely only if the signed contract permits it. Employee rules do not apply to contractors. A contractor is a person hired for a fixed task and a fixed period, and the terms of the work are written in the signed contract. If the contract says the work must be done on site, the contractor must attend the office for the full working day. Remote work is allowed only when the contract states it clearly, including the days and hours. Employees, by contrast, may request remote work with manager approval. An employee who works from home must stay available on the office chat during working hours and attend meetings online. A contractor cannot claim employee benefits, and an employee cannot claim contractor rates. When a contract ends, remote access is closed on the same day. Any dispute about remote work is settled from the contract text, not from the employee handbook. Managers must check the contract before approving a remote request from a contractor, because a wrong approval can break the agreement.",
  "printer": "For blank printer pages, check ink levels and run the cleaning utility. Print a test page afterwards. A printer that has power but produces blank pages usually has empty ink cartridges, a blocked print head, or a driver setting that prints with the wrong paper size. Open the printer settings on your computer and check the ink level before you open the printer, because a cartridge that looks full from the outside can still be empty inside. If the ink level is low, replace the cartridge with the model number printed on the old one. After replacing the cartridge, run the cleaning utility from the printer software, and wait until it finishes, because stopping it early can leave the print head blocked, and a blocked print head is the most common reason a printer keeps producing blank pages even after a new cartridge is fitted, so the cleaning step must not be skipped. Then print a test page to confirm the problem is fixed. If the test page is still blank, check the cable and the driver version. Do not open the print head by hand.",
  "employee": "Employees may request remote work with manager approval. A full-time employee can ask to work from home for up to two days each week, and the request must be sent to the manager at least one day before. The manager approves the request after checking the team schedule and the work that is due that week. An employee working from home must keep the office chat open, answer calls during working hours, and join every scheduled meeting online. Work that needs office equipment, such as the lab machines or the printer room, cannot be done from home. The manager can cancel remote work on any day when the team needs the employee in the office. Remote work is a permission, not a right, and it can be withdrawn after two missed meetings or late replies. New employees complete their first month on site before they can request remote work. Contractors follow their own contract terms, and this policy does not cover them. Records of remote days are kept by the manager."
}

evidence = {
  "late": "Illness extensions require medical evidence before the deadline.",
  "password": "Reset a forgotten password using the recovery link.",
  "assignment": "Upload assignments through the learning portal.",
  "contractor": "Contractors can work remotely only if the signed contract permits it.",
  "printer": "For blank printer pages, check ink levels and run the cleaning utility.",
  "employee": "Employees may request remote work with manager approval."
}


def word_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]


def windows(text, size=40, overlap=10, offset=0):
    if size <= 0 or not 0 <= overlap < size:
        raise ValueError("Require size > 0 and 0 <= overlap < size")
    spans = word_spans(text)
    result = []
    for i in range(0, len(spans), size - overlap):
        j = min(i + size, len(spans))
        result.append((offset + spans[i][0], offset + spans[j - 1][1]))
        if j == len(spans):
            break
    return result


def sentence_chunks(text, size=40):
    if size <= 0:
        raise ValueError("size must be positive")
    cuts = [0] + [m.end() for m in re.finditer(r"[.!?]\s+", text)] + [len(text)]
    out, pending, count = [], None, 0
    for a, b in zip(cuts, cuts[1:]):
        segment = text[a:b]
        n = len(segment.split())
        if not n:
            continue
        if n > size:
            if pending is not None:
                out.append(pending)
            out.extend(windows(segment, size, 0, a))
            pending, count = None, 0
        elif pending is not None and count + n <= size:
            pending, count = (pending[0], b), count + n
        else:
            if pending is not None:
                out.append(pending)
            pending, count = (a, b), n
    if pending is not None:
        out.append(pending)
    return out


def make_chunks(docs, strategy):
    result = []
    for doc_id, text in docs.items():
        spans = sentence_chunks(text) if strategy == "sentence" else windows(text, 40, int(strategy))
        for i, (a, b) in enumerate(spans):
            result.append(dict(id=f"{doc_id}:{i}", doc=doc_id, start=a, end=b, text=text[a:b]))
    return result


for strategy in ["0", "10", "sentence"]:
    chunks = make_chunks(docs, strategy)
    total = sum(len(c["text"].split()) for c in chunks)
    original = sum(len(t.split()) for t in docs.values())
    sizes = [len(c["text"].split()) for c in chunks]
    kept = [d for d in docs if any(evidence[d] in c["text"] for c in chunks if c["doc"] == d)]
    print(strategy, "chunks:", len(chunks), "| size min/max:", min(sizes), max(sizes), "| repeated words:", total - original, "| evidence kept whole:", kept)

0 chunks: 31 | size min/max: 11 40 | repeated words: 0 | evidence kept whole: ['late', 'password', 'assignment', 'contractor', 'printer', 'employee']
10 chunks: 37 | size min/max: 21 40 | repeated words: 310 | evidence kept whole: ['late', 'password', 'assignment', 'contractor', 'printer', 'employee']
sentence chunks: 37 | size min/max: 15 40 | repeated words: 0 | evidence kept whole: ['late', 'password', 'assignment', 'contractor', 'printer', 'employee']


In [5]:
print("Empty text:", windows(""))
print("Short document:", windows("short text"))
try:
    windows("a b", 2, 2)
except ValueError:
    print("Invalid overlap: ValueError raised")
long_text = " ".join(["word"] * 95) + "."
print("95-word sentence split into:", [len(long_text[a:b].split()) for a, b in sentence_chunks(long_text)])
for strategy in ["0", "10", "sentence"]:
    for chunk in make_chunks({"example": "Reset your password. Keep recovery codes private."}, strategy):
        assert chunk["text"] == "Reset your password. Keep recovery codes private."[chunk["start"]:chunk["end"]]
print("Every chunk equals its source slice.")

Empty text: []
Short document: [(0, 10)]
Invalid overlap: ValueError raised
95-word sentence split into: [40, 40, 15]
Every chunk equals its source slice.


In [6]:
records = make_chunks(docs, "sentence")
paraphrases = ["How can I regain access to my account?", "I was sick, can I hand my work in late?", "The printer gives empty sheets, what should I check?"]
for backend in ["tfidf", "embedding"]:
    engine = Retriever(records, backend)
    print("Backend:", backend)
    for query in paraphrases:
        print("Query:", query)
        show_hits(engine.search(query, k=2))

Backend: tfidf
Query: How can I regain access to my account?
employee:0 score=0.202
Employees may request remote work with manager approval. A full-time employee can ask to work from home for up to two days each week, and the request must be sent to the manager at least one day before. 
password:3 score=0.179
After three failed login attempts, the account locks for thirty minutes, and you must wait before trying again. If the account stays locked, contact the IT help desk with your student ID card. 
Query: I was sick, can I hand my work in late?
employee:3 score=0.217
Remote work is a permission, not a right, and it can be withdrawn after two missed meetings or late replies. New employees complete their first month on site before they can request remote work. 
late:1 score=0.173
Work submitted after the deadline loses ten percent of its marks for each day it is late, and work submitted more than five days late receives no marks at all. 
Query: The printer gives empty sheets, what shoul

In [7]:
queries = [{'query': 'What is required for illness extensions?', 'doc': 'late', 'split': 'dev', 'evidence': 'Illness extensions require medical evidence before the deadline.'}, {'query': 'How do I reset a forgotten password?', 'doc': 'password', 'split': 'dev', 'evidence': 'Reset a forgotten password using the recovery link.'}, {'query': 'Where do I upload assignments?', 'doc': 'assignment', 'split': 'dev', 'evidence': 'Upload assignments through the learning portal.'}, {'query': 'Can contractors work remotely?', 'doc': 'contractor', 'split': 'dev', 'evidence': 'Contractors can work remotely only if the signed contract permits it.'}, {'query': 'What do I do for blank printer pages?', 'doc': 'printer', 'split': 'test', 'evidence': 'For blank printer pages, check ink levels and run the cleaning utility.'}, {'query': 'How can I recover my login access?', 'doc': 'password', 'split': 'test', 'evidence': 'Reset a forgotten password using the recovery link.'}, {'query': 'How should I hand in my coursework?', 'doc': 'assignment', 'split': 'test', 'evidence': 'Upload assignments through the learning portal.'}, {'query': 'What proof is needed when I was ill?', 'doc': 'late', 'split': 'test', 'evidence': 'Illness extensions require medical evidence before the deadline.'}, {'query': "What are tomorrow's flight prices?", 'doc': None, 'split': 'test', 'evidence': None}, {'query': 'Who won the football match?', 'doc': None, 'split': 'test', 'evidence': None}]


def evaluate(scores, chunks, items):
    hits, reciprocal = [], []
    for row, q in zip(scores, items):
        if q["doc"] is None:
            continue
        top = np.argsort(-row, kind="stable")[:3]
        good = [rank for rank, i in enumerate(top, 1) if chunks[i]["doc"] == q["doc"] and q["evidence"] in chunks[i]["text"]]
        hits.append(bool(good))
        reciprocal.append(1 / good[0] if good else 0)
    return (float(np.mean(hits)), float(np.mean(reciprocal))) if hits else (None, None)


import csv
query_texts = [q["query"] for q in queries]
logs = []
for strategy in ["0", "10", "sentence"]:
    chunks = make_chunks(docs, strategy)
    texts = [c["text"] for c in chunks]
    lengths = [len(model.tokenizer(t, truncation=False)["input_ids"]) for t in texts + query_texts]
    print(strategy, "chunks:", len(chunks), "| max tokens:", max(lengths), "| limit:", model.max_seq_length)
    for method in ["tfidf", "embedding"]:
        t = time.perf_counter()
        if method == "tfidf":
            vectorizer = TfidfVectorizer()
            C = vectorizer.fit_transform(texts)
            Q = vectorizer.transform(query_texts)
        else:
            C = model.encode(texts, normalize_embeddings=True)
            Q = model.encode(query_texts, normalize_embeddings=True)
        encode_seconds = time.perf_counter() - t
        t = time.perf_counter()
        scores = (Q @ C.T).toarray() if method == "tfidf" else Q @ C.T
        search_seconds = time.perf_counter() - t
        for split in ["dev", "test"]:
            indices = [i for i, q in enumerate(queries) if q["split"] == split]
            result = evaluate(scores[indices], chunks, [queries[i] for i in indices])
            print(strategy, method, split, "Hit@3/MRR@3:", tuple(round(x, 3) for x in result), "| encode:", round(encode_seconds, 3), "search:", round(search_seconds, 4))
        for row, q in zip(scores, queries):
            for rank, i in enumerate(np.argsort(-row, kind="stable")[:3], 1):
                c = chunks[i]
                logs.append(dict(strategy=strategy, method=method, query=q["query"], split=q["split"], rank=rank, score=round(float(row[i]), 3), chunk_id=c["id"], doc=c["doc"], start=c["start"], end=c["end"], text=c["text"], answerable=q["doc"] is not None))
with open("retrieval_log.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(logs[0]))
    writer.writeheader()
    writer.writerows(logs)
print("retrieval_log.csv rows:", len(logs))

0 chunks: 31 | max tokens: 53 | limit: 256
0 tfidf dev Hit@3/MRR@3: (1.0, 1.0) | encode: 0.005 search: 0.0003
0 tfidf test Hit@3/MRR@3: (0.25, 0.25) | encode: 0.005 search: 0.0003
0 embedding dev Hit@3/MRR@3: (1.0, 0.875) | encode: 1.009 search: 0.0016
0 embedding test Hit@3/MRR@3: (1.0, 0.75) | encode: 1.009 search: 0.0016
10 chunks: 37 | max tokens: 53 | limit: 256
10 tfidf dev Hit@3/MRR@3: (1.0, 1.0) | encode: 0.005 search: 0.0003
10 tfidf test Hit@3/MRR@3: (0.25, 0.25) | encode: 0.005 search: 0.0003
10 embedding dev Hit@3/MRR@3: (1.0, 1.0) | encode: 1.306 search: 0.0001
10 embedding test Hit@3/MRR@3: (1.0, 0.75) | encode: 1.306 search: 0.0001
sentence chunks: 37 | max tokens: 49 | limit: 256
sentence tfidf dev Hit@3/MRR@3: (1.0, 1.0) | encode: 0.012 search: 0.0004
sentence tfidf test Hit@3/MRR@3: (0.25, 0.25) | encode: 0.012 search: 0.0004
sentence embedding dev Hit@3/MRR@3: (1.0, 0.875) | encode: 1.687 search: 0.0001
sentence embedding test Hit@3/MRR@3: (1.0, 0.708) | encode: 1.68

In [8]:
chunks = make_chunks(docs, "10")
texts = [c["text"] for c in chunks]
C = model.encode(texts, normalize_embeddings=True)
Q = model.encode(query_texts, normalize_embeddings=True)
scores = Q @ C.T
for qi, q in enumerate(queries):
    print("Query:", q["query"])
    for rank, i in enumerate(np.argsort(-scores[qi], kind="stable")[:3], 1):
        c = chunks[i]
        print(rank, c["id"], f"score={float(scores[qi][i]):.3f}", "chars", c["start"], c["end"])
        print(c["text"])

Query: What is required for illness extensions?
1 late:0 score=0.582 chars 0 263
Late submission is normally penalized. Illness extensions require medical evidence before the deadline. Every assignment in this course has a fixed deadline published in the learning portal at the start of the semester. Work submitted after the deadline loses ten
2 late:2 score=0.552 chars 366 613
marks at all. A student who was ill during the deadline period may request an extension, but the request must reach the course coordinator before the original deadline passes. The request must include a medical certificate from a registered doctor
3 late:5 score=0.509 chars 904 1142
copy of their medical evidence and the approval message. If an extension is refused, the student may appeal to the department within seven days. Group assignments follow the same rule, and one member being ill does not extend the deadline
Query: How do I reset a forgotten password?
1 password:0 score=0.652 chars 0 244
Reset a forgotte

In [9]:
bot_engine = Retriever(make_chunks(docs, "10"), "embedding")


def show(query):
    print("Question:", query)
    hits = bot_engine.search(query)
    if not hits:
        print("No candidate evidence; rephrase or contact a human.")
        return []
    for number, (r, score) in enumerate(hits, 1):
        print(number, r["id"], f"{score:.3f}", "characters", r["start"], r["end"])
    return hits


def answer(query, choice):
    hits = bot_engine.search(query)
    if not choice or not 1 <= choice <= len(hits):
        print("No answer selected. Rephrase or contact a human.")
        return
    r, score = hits[choice - 1]
    print("Candidate evidence; check it before using:")
    print(r["text"])
    print("Source:", r["doc"], "characters", r["start"], r["end"])


show("How can I recover my login access?")
answer("How can I recover my login access?", 2)
show("Who won the football match?")
answer("Who won the football match?", 0)

Question: How can I recover my login access?
1 password:3 0.557 characters 522 747
2 password:0 0.548 characters 0 244
3 password:4 0.524 characters 692 900
Candidate evidence; check it before using:
Reset a forgotten password using the recovery link. Never share recovery codes. Every student account uses the university email address as its username, and the password is set by the student during the first login. If you forget your password,
Source: password characters 0 244
Question: Who won the football match?
1 password:1 0.116 characters 184 407
2 password:0 0.098 characters 0 244
3 employee:1 0.081 characters 163 386
No answer selected. Rephrase or contact a human.


In [ ]:
print("Evidence assistant. Type exit to stop.")
while True:
    try:
        query = input("Question: ")
    except Exception:
        break
    if query.strip().lower() == "exit":
        break
    hits = show(query)
    if not hits:
        continue
    try:
        chosen = input("Choose 1-3 to inspect; Enter if none apply: ")
    except Exception:
        break
    if not chosen.isdigit() or not 1 <= int(chosen) <= len(hits):
        print("No answer selected. Rephrase or contact a human.")
        continue
    answer(query, int(chosen))

Evidence assistant. Type exit to stop.
Question: hi
1 password:1 0.131 characters 184 407
2 employee:1 0.120 characters 163 386
3 password:0 0.115 characters 0 244
Candidate evidence; check it before using:
Reset a forgotten password using the recovery link. Never share recovery codes. Every student account uses the university email address as its username, and the password is set by the student during the first login. If you forget your password,
Source: password characters 0 244
